In [1]:
import json
import os
from urllib.error import URLError
from urllib.parse import urlencode
from urllib.request import urlopen

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnableConfig
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_deepseek import ChatDeepSeek
from rich import print as rprint

load_dotenv(override=True)

api_key = os.getenv("DEEPSEEK_API_KEY")
api_base = os.getenv("DEEPSEEK_API_BASE")

### 模型初始化的参数

In [2]:

chat_deepseek =  ChatDeepSeek(
    model="deepseek-v4-flash",    # model_name='deepseek-flush', # langchain模型并没有声明deepseek-flush的模型信息
)

rprint(chat_deepseek.profile)

{
    'name': 'DeepSeek V4 Flash',
    'release_date': '2026-04-24',
    'last_updated': '2026-04-24',
    'open_weights': True,
    'max_input_tokens': 1000000,
    'max_output_tokens': 384000,
    'text_inputs': True,
    'image_inputs': False,
    'audio_inputs': False,
    'video_inputs': False,
    'text_outputs': True,
    'image_outputs': False,
    'audio_outputs': False,
    'video_outputs': False,
    'reasoning_output': True,
    'tool_calling': True,
    'structured_output': True,
    'attachment': False,
    'temperature': True
}

### init_chat_model的模型参数

In [4]:
chat_model = init_chat_model(api_key = api_key,api_base =  api_base,model="deepseek-v4-flash")

rprint(chat_model.profile)

{
    'name': 'DeepSeek V4 Flash',
    'release_date': '2026-04-24',
    'last_updated': '2026-04-24',
    'open_weights': True,
    'max_input_tokens': 1000000,
    'max_output_tokens': 384000,
    'text_inputs': True,
    'image_inputs': False,
    'audio_inputs': False,
    'video_inputs': False,
    'text_outputs': True,
    'image_outputs': False,
    'audio_outputs': False,
    'video_outputs': False,
    'reasoning_output': True,
    'tool_calling': True,
    'structured_output': True,
    'attachment': False,
    'temperature': True
}

### model_kwargs  透传模型支持，但是langchain没有列出的字段(标准OpenAI的API参数)

In [9]:
@tool
def get_weather(city: str) -> str:
    """获取指定城市的实时天气。"""
    try:
        location_query = urlencode({"name": city, "count": 1, "language": "zh", "format": "json"})
        with urlopen(f"https://geocoding-api.open-meteo.com/v1/search?{location_query}", timeout=10) as response:
            location_data = json.load(response)

        locations = location_data.get("results", [])
        if not locations:
            return f"没有找到城市：{city}"

        location = locations[0]
        weather_query = urlencode({
            "latitude": location["latitude"],
            "longitude": location["longitude"],
            "current": "temperature_2m,apparent_temperature,relative_humidity_2m,weather_code,wind_speed_10m",
            "timezone": "auto",
        })
        with urlopen(f"https://api.open-meteo.com/v1/forecast?{weather_query}", timeout=10) as response:
            weather_data = json.load(response)

        return json.dumps({
            "城市": location["name"],
            "地区": location.get("admin1"),
            "国家": location.get("country"),
            "当前天气": weather_data["current"],
            "单位": weather_data["current_units"],
        }, ensure_ascii=False)
    except (URLError, TimeoutError, json.JSONDecodeError, KeyError) as exc:
        return f"天气服务请求失败：{exc}"


weather_tool_schema = convert_to_openai_tool(get_weather)
chat_deepseek = ChatDeepSeek(
    model="deepseek-chat",
    model_kwargs={"tools": [weather_tool_schema]},
)

messages = [("user", "今天上海天气如何？")]
response = chat_deepseek.invoke(messages)
rprint(response.tool_calls)


AIMessage(
    content='',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 38,
            'prompt_tokens': 270,
            'total_tokens': 308,
            'completion_tokens_details': None,
            'prompt_tokens_details': {
                'audio_tokens': None,
                'cache_write_tokens': None,
                'cached_tokens': 128,
                'image_tokens': None,
                'text_tokens': None
            },
            'prompt_cache_hit_tokens': 128,
            'prompt_cache_miss_tokens': 142
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-flash',
        'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
        'id': '20d03b82-7208-4fa9-b073-30f05f95b1b6',
        'finish_reason': 'tool_calls',
        'logprobs': None
    },
    id='lc_run--01a0c33c-79e6-7123-bf14-f63dc5b0779b-0',
    tool_calls=[
        {
            'name': 'get_weather',
            'args': {'city': '上海'},
            'id': 'call_00_eBAH9hWI0SGQcm27On8p6032',
            'type': 'tool_call'
        }
    ],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 270,
        'output_tokens': 38,
        'total_tokens': 308,
        'input_token_details': {'cache_read': 128},
        'output_token_details': {}
    }
)

[{'name': 'get_weather', 'args': {'city': '上海'}, 'id': 'call_00_eBAH9hWI0SGQcm27On8p6032', 'type': 'tool_call'}]

### extra_body
* 非openai，而是提供商特有的非标准参数（ DeepSeek 的 thinking）

In [9]:
chat_deepseek = ChatDeepSeek(
    # 未开启thinking model
    model="deepseek-chat",
    # DeepSeek 官方文档明确说明deepseek-flash Thinking Mode 默认开启、默认强度为 high,
    # model="deepseek-flash",
    extra_body={"thinking": {"type": "enabled"}},
)
response = chat_deepseek.invoke("你是谁，一句话回答")
print(response.content_blocks)
rprint(response.content_blocks)  # 包含思考过程additional_kwargs.reasoning_content

[{'type': 'reasoning', 'reasoning': '我们需要回答用户“你是谁，一句话回答”。需要一句话。我是谁？我是AI助手，由深度求索创造的DeepSeek？根据系统提示未给名字。通常回答“我是DeepSeek，由深度求索公司创造的AI助手。” 一句话。需要中文。确保一句话。'}, {'type': 'text', 'text': '我是由深度求索公司创造的AI助手，叫DeepSeek。'}]


[
    {
        'type': 'reasoning',
        'reasoning': 
'我们需要回答用户“你是谁，一句话回答”。需要一句话。我是谁？我是AI助手，由深度求索创造的DeepSeek？根据系统提示未给名
字。通常回答“我是DeepSeek，由深度求索公司创造的AI助手。” 一句话。需要中文。确保一句话。'
    },
    {'type': 'text', 'text': '我是由深度求索公司创造的AI助手，叫DeepSeek。'}
]

### config  指定调用时的参数
* 单次调用的指定参数

In [12]:
chat_model = init_chat_model(
    model="deepseek:deepseek-v4-pro",
    api_base=api_base,
    api_key=api_key,
    max_tokens=1000,
    temperature=0.2,
    configurable_fields=["model", "temperature"],  # 允许config修改的配置
)
my_config = RunnableConfig(configurable={"model": "deepseek:deepseek-v4-flash"})

response = chat_model.invoke("简短介绍下自己", config=my_config)
rprint(response)
print(response.content)

AIMessage(
    content='你好，我是 DeepSeek，由深度求索开发的 AI 
助手。可以帮你解答问题、写作、翻译、编程、总结和分析等。有需要随时告诉我！',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': '我们需要回答用户中文请求：“简短介绍下自己”。作为 AI 
助手，应该简短介绍自己。需要以中文回答，可能提及我是由深度求索开发的 AI 
助手，能帮助回答问题、写作、翻译、编程、分析等。注意不要过度。应该简短。需要确保不声称有身体/情感？可以自然。用户要
求简短介绍下自己。可以答：你好，我是 DeepSeek，由深度求索开发的 AI 
助手。我可以帮你解答问题、写作、翻译、编程、总结等。有什么需要随时说。但注意当前模型身份？系统没有明确说我是 
DeepSeek？实际上通常这个 AI 是 
DeepSeek。需要遵守。可以简短。也许提“支持中英文，多轮对话”。不要提知识截止日期？可不说。用户只是要求介绍自己。用友
好语气。保持简短。最终直接回答。'
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 220,
            'prompt_tokens': 34,
            'total_tokens': 254,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 183,
                'rejected_prediction_tokens': None,
                'text_tokens': None
            },
            'prompt_tokens_details': {
                'audio_tokens': None,
                'cache_write_tokens': None,
                'cached_tokens': 0,
                'image_tokens': None,
                'text_tokens': None
            },
            'prompt_cache_hit_tokens': 0,
            'prompt_cache_miss_tokens': 34
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-flash',
        'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
        'id': 'ce81ed6a-4ad2-4937-88ba-edfb47933d8a',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--01a0c428-0ee2-7540-ad3e-60de3028eac8-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 34,
        'output_tokens': 220,
        'total_tokens': 254,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 183}
    }
)

你好，我是 DeepSeek，由深度求索开发的 AI 助手。可以帮你解答问题、写作、翻译、编程、总结和分析等。有需要随时告诉我！
